In [1]:
pip install pandas requests mplsoccer matplotlib

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
import pandas as pd
import time

all_records = []
total_pages = 19 
base_url = "https://ss2.si-ab.com/opta-stats/27018/players/season/?locale=da&sortBy=goals&aggregateBy=sum&page="

for page in range(1, total_pages + 1):
    response = requests.get(f"{base_url}{page}")
    if response.status_code == 200:
        data = response.json()
        all_records.extend(data['records'])
        print(f"Fetched page {page}/{total_pages}")
    time.sleep(1) # Keep this for "Safe Scraping"

df_raw = pd.json_normalize(all_records)

In [ ]:
# 1. Filter for Midfielders with 900+ minutes
# This changes the "Peer Group" Carstensen is ranked against
df_midfielders = df_raw[(df_raw['position'] == 'midfielder') & (df_raw['stats.mins_played'] >= 900)].copy()

# 2. Add Carstensen back into this group manually so he can be ranked against them
# This is a key IT Architect move: appending a specific record to a filtered dataset
carstensen_row = df_raw[df_raw['name'] == "Rasmus Carstensen"]
df_compare = pd.concat([df_midfielders, carstensen_row]).drop_duplicates().copy()

# 3. Clean and Rank
df_compare = df_compare.fillna(0)
for metric in metrics:
    df_compare[f"{metric}_rank"] = df_compare[metric].rank(pct=True) * 100

# 4. Extract his new ranks
player_stats = df_compare[df_compare['name'] == "Rasmus Carstensen"]

In [ ]:
from mplsoccer import PyPizza
import matplotlib.pyplot as plt

# 1. Setup the labels
labels = ["Pass %", "Prog. Carries", "Duels Won", "Top Speed", "Interceptions", "Tackles"]

# 2. Initialize the Pizza
bakery = PyPizza(
    params=labels,
    background_color="#FFFFFF",
    straight_line_color="#222222",
    last_circle_lw=1,
    other_circle_lw=1,
    other_circle_color="#D1D1D1",
)

# 3. Plot the data (Removed 'value_offset' to fix the error)
fig, ax = bakery.make_pizza(
    values,
    figsize=(8, 8),
    param_location=118,           # Moved even further out to 118
    slice_colors=["#BA0C2F"] * 6, # AGF Red
    value_colors=["#000000"] * 6,
    value_bck_colors=["#FFFFFF"] * 6,
    kwargs_params=dict(size=12),  # Control label font size
    kwargs_values=dict(size=11),  # Control value font size
)

plt.show()